# Leukemia 4-Model Retraining — Resumable, Checkpointed (GPU)
**MobileNetV2 / Xception / ResNet50 / DenseNet121 — matches the architecture already in the paper**

This fixes the earlier run, which was cut off at epoch 3 of 30 for every model (session interruption, not
early stopping). This version:
- Saves a checkpoint **and** the running history to **Google Drive after every single epoch**
- On restart, **automatically resumes** each model from its last saved epoch — no lost progress
- Uses the exact same architecture/hyperparameters as your original notebook (label smoothing 0.05,
  lr=1e-4, last-20-layers fine-tuning, ReduceLROnPlateau, EarlyStopping patience=6) so results stay
  directly comparable to the paper's Methods section

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine). Then Runtime -> Run all.
If Colab disconnects, just reopen and Run all again — it will pick up from the last completed epoch.

## 1. Mount Drive (for crash-proof checkpointing) + GPU check

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))
assert len(tf.config.list_physical_devices('GPU')) > 0, "No GPU detected — set Runtime > Change runtime type > GPU"

import os
DRIVE_ROOT = "/content/drive/MyDrive/leukemia_4model_run"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/history", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
print("Persistent run folder:", DRIVE_ROOT)

Mounted at /content/drive
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Persistent run folder: /content/drive/MyDrive/leukemia_4model_run


## 2. Setup: Kaggle API + download dataset (skipped automatically if already downloaded to Drive)

In [2]:
DATA_ROOT = "/content/drive/MyDrive/leukemia_4model_run/data"
FLAT_DIR = f"{DATA_ROOT}/flattened"
SPLITS_DIR = f"{DATA_ROOT}/splits"
CLASS_NAMES = ["all", "hem"]

if not os.path.exists(FLAT_DIR) or not os.listdir(FLAT_DIR):
    os.makedirs(DATA_ROOT, exist_ok=True)
    from google.colab import files
    print("Upload your kaggle.json (Kaggle -> Settings -> Create New API Token)")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    import shutil
    shutil.copy(list(uploaded.keys())[0], "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    RAW_DIR = f"{DATA_ROOT}/raw"
    os.makedirs(RAW_DIR, exist_ok=True)
    !kaggle datasets download -d andrewmvd/leukemia-classification -p {RAW_DIR} --unzip

    from pathlib import Path
    for c in CLASS_NAMES:
        (Path(FLAT_DIR) / c).mkdir(parents=True, exist_ok=True)
    fold_root = Path(RAW_DIR) / "C-NMC_Leukemia" / "training_data"
    for fold_dir in sorted(fold_root.glob("fold_*")):
        for c in CLASS_NAMES:
            src_dir = fold_dir / c
            if not src_dir.exists():
                continue
            for img_path in src_dir.iterdir():
                if img_path.is_file():
                    dest = Path(FLAT_DIR) / c / f"{fold_dir.name}_{img_path.name}"
                    if not dest.exists():
                        shutil.copy(img_path, dest)
    print("Dataset flattened to Drive — will persist across sessions, no re-download needed next time.")
else:
    print("Flattened dataset already exists on Drive — skipping download.")

Upload your kaggle.json (Kaggle -> Settings -> Create New API Token)


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/leukemia-classification
License(s): other
100% 867M/867M [00:08<00:00, 112MB/s] 

Dataset flattened to Drive — will persist across sessions, no re-download needed next time.


## 3. Build (or load) the shared 70/15/15 split — saved to Drive so it's identical across resumed sessions

In [3]:
import json, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

SPLIT_FILE = f"{DATA_ROOT}/split_paths.json"
RANDOM_SEED = 42

if os.path.exists(SPLIT_FILE):
    with open(SPLIT_FILE) as f:
        split = json.load(f)
    train_p, train_y = np.array(split["train_p"]), np.array(split["train_y"])
    val_p, val_y     = np.array(split["val_p"]),   np.array(split["val_y"])
    test_p, test_y   = np.array(split["test_p"]),  np.array(split["test_y"])
    print("Loaded existing split from Drive (guarantees identical split across resumed sessions).")
else:
    all_paths, all_labels = [], []
    for idx, c in enumerate(CLASS_NAMES):
        for p in Path(f"{FLAT_DIR}/{c}").iterdir():
            if p.is_file():
                all_paths.append(str(p)); all_labels.append(idx)
    all_paths, all_labels = np.array(all_paths), np.array(all_labels)

    train_p, temp_p, train_y, temp_y = train_test_split(
        all_paths, all_labels, train_size=0.70, stratify=all_labels, random_state=RANDOM_SEED)
    val_p, test_p, val_y, test_y = train_test_split(
        temp_p, temp_y, train_size=0.5, stratify=temp_y, random_state=RANDOM_SEED)

    with open(SPLIT_FILE, "w") as f:
        json.dump({"train_p": train_p.tolist(), "train_y": train_y.tolist(),
                    "val_p": val_p.tolist(), "val_y": val_y.tolist(),
                    "test_p": test_p.tolist(), "test_y": test_y.tolist()}, f)
    print("Built new split and saved to Drive.")

print(f"Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}")

Built new split and saved to Drive.
Train: 7462 | Val: 1599 | Test: 1600


## 4. Data pipeline (matches original notebook: per-backbone preprocessing, 224x224)

In [4]:
from tensorflow.keras import layers
from tensorflow.keras.applications import mobilenet_v2, xception, resnet50, densenet

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

PREPROCESS_MAP = {
    "MobileNetV2": mobilenet_v2.preprocess_input,
    "Xception": xception.preprocess_input,
    "ResNet50": resnet50.preprocess_input,
    "DenseNet121": densenet.preprocess_input,
}
BACKBONE_MAP = {
    "MobileNetV2": tf.keras.applications.MobileNetV2,
    "Xception": tf.keras.applications.Xception,
    "ResNet50": tf.keras.applications.ResNet50,
    "DenseNet121": tf.keras.applications.DenseNet121,
}

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.0417),   # ~15 degrees
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

def make_dataset(paths, labels, batch_size, training):
    labels_oh = tf.keras.utils.to_categorical(labels, num_classes=2)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_oh))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, IMAGE_SIZE)
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=RANDOM_SEED)
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_raw = make_dataset(train_p, train_y, BATCH_SIZE, training=True)
val_ds_raw   = make_dataset(val_p,   val_y,   BATCH_SIZE, training=False)
test_ds_raw  = make_dataset(test_p,  test_y,  BATCH_SIZE, training=False)

## 5. Model builder — identical architecture to the original notebook (partial fine-tuning, last 20 layers trainable)

In [5]:
LEARNING_RATE = 1e-4

def build_model(model_name, fine_tune_layers=20):
    inputs = layers.Input(shape=(224, 224, 3), name="input_image")
    prep_fn = PREPROCESS_MAP[model_name]
    x = layers.Lambda(prep_fn, name=f"{model_name.lower()}_prep")(inputs)

    base = BACKBONE_MAP[model_name](weights="imagenet", include_top=False, input_shape=(224, 224, 3), input_tensor=x)
    base.trainable = True
    if fine_tune_layers > 0 and len(base.layers) > fine_tune_layers:
        for layer in base.layers[:-fine_tune_layers]:
            layer.trainable = False

    feat = layers.GlobalAveragePooling2D(name="gap")(base.output)
    bn = layers.BatchNormalization(name="bn")(feat)
    drop1 = layers.Dropout(0.3, name="drop1")(bn)
    dense = layers.Dense(128, activation="relu", name="dense128")(drop1)
    drop2 = layers.Dropout(0.2, name="drop2")(dense)
    out = layers.Dense(2, activation="softmax", name="pred")(drop2)

    model = tf.keras.Model(inputs=inputs, outputs=out, name=f"{model_name}_Classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")]
    )
    return model

## 6. Resumable training loop
For each model: if a checkpoint + history already exist on Drive, load the model and continue from the
next epoch; otherwise start fresh. History is appended and saved to Drive after **every epoch**, so a
disconnect at any point loses at most the epoch in progress.

In [ ]:
import time

EPOCHS = 30
MODELS_TO_TRAIN = ["MobileNetV2", "Xception", "ResNet50", "DenseNet121"]

class SaveHistoryEveryEpoch(tf.keras.callbacks.Callback):
    """Appends this epoch's metrics to the Drive history file after every epoch completes."""
    def __init__(self, history_path, start_epoch=0):
        super().__init__()
        self.history_path = history_path
        self.epoch_offset = start_epoch

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if os.path.exists(self.history_path):
            with open(self.history_path) as f:
                hist = json.load(f)
        else:
            hist = {k: [] for k in list(logs.keys()) + ["duration_min"]}
        for k, v in logs.items():
            hist.setdefault(k, []).append(float(v))
        hist.setdefault("duration_min", []).append(round((time.time() - self._epoch_t0) / 60.0, 3))
        with open(self.history_path, "w") as f:
            json.dump(hist, f, indent=2)

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_t0 = time.time()


for name in MODELS_TO_TRAIN:
    print("=" * 60, f"\n{name}\n", "=" * 60)
    ckpt_path = f"{DRIVE_ROOT}/checkpoints/{name}_last.keras"
    best_path = f"{DRIVE_ROOT}/checkpoints/{name}_best.keras"
    hist_path = f"{DRIVE_ROOT}/history/history_{name}.json"

    if os.path.exists(ckpt_path) and os.path.exists(hist_path):
        with open(hist_path) as f:
            prior_hist = json.load(f)
        start_epoch = len(prior_hist.get("loss", []))
        if start_epoch >= EPOCHS:
            print(f"{name} already completed {start_epoch} epochs — skipping.")
            continue
        print(f"Resuming {name} from epoch {start_epoch}/{EPOCHS}")
        model = tf.keras.models.load_model(ckpt_path)
    else:
        print(f"Starting {name} fresh")
        model = build_model(name)
        start_epoch = 0

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=False, verbose=0),
        tf.keras.callbacks.ModelCheckpoint(best_path, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        SaveHistoryEveryEpoch(hist_path, start_epoch=start_epoch),
    ]

    model.fit(train_ds_raw, validation_data=val_ds_raw, epochs=EPOCHS,
              initial_epoch=start_epoch, callbacks=callbacks, verbose=1)

print("\nAll models done (or resumed to completion). Re-run this cell after any disconnect to continue.")

MobileNetV2
Starting MobileNetV2 fresh
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 0s 655ms/step - accuracy: 0.7006 - auc: 0.7658 - loss: 0.7918
Epoch 1: val_accuracy improved from None to 0.31770, saving model to /content/drive/MyDrive/leukemia_4model_run/checkpoints/MobileNetV2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/leukemia_4model_run/checkpoints/MobileNetV2_best.keras
234/234 ━━━━━━━━━━━━━━━━━━━━ 257s 778ms/step - accuracy: 0.7361 - auc: 0.8121 - loss: 0.6947 - val_accuracy: 0.3177 - val_auc: 0.3857 - val_loss: 3.2855 - learning_rate: 1.0000e-04
Epoch 2/30
234/234 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.7693 - auc: 0.8472 - loss: 0.6001
Epoch 2: val_accuracy improved from 0.31770 to 0.53596, saving model to /content/drive/MyDrive/leukemia_4model_run/checkpoints/MobileNetV2_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/leukemia_4model_run/checkpoints/MobileNetV2_best.keras
234/

## 7. Evaluate all 4 models on the shared test set (uses each model's `_best` checkpoint)

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_auc_score, precision_recall_fscore_support, accuracy_score)
from scipy import stats as scistats

y_true = test_y
all_results = {}
pred_labels = {}
pred_probs = {}

for name in MODELS_TO_TRAIN:
    best_path = f"{DRIVE_ROOT}/checkpoints/{name}_best.keras"
    model = tf.keras.models.load_model(best_path)

    t0 = time.time()
    probs = model.predict(test_ds_raw, verbose=0)
    inf_time = (time.time() - t0) / len(test_p)
    pred_probs[name] = probs
    pred_labels[name] = np.argmax(probs, axis=1)

    acc = accuracy_score(y_true, pred_labels[name])
    prec, rec, f1, support = precision_recall_fscore_support(y_true, pred_labels[name], average=None, labels=[0, 1])
    macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(y_true, pred_labels[name], average="macro")
    auc_val = roc_auc_score(y_true, probs[:, 1])
    cm = confusion_matrix(y_true, pred_labels[name])
    num_params = model.count_params()

    all_results[name] = {
        "accuracy": float(acc), "macro_precision": float(macro_prec), "macro_recall": float(macro_rec),
        "macro_f1": float(macro_f1), "roc_auc": float(auc_val),
        "per_class": {CLASS_NAMES[i]: {"precision": float(prec[i]), "recall": float(rec[i]),
                                         "f1_score": float(f1[i]), "support": int(support[i])} for i in range(2)},
        "confusion_matrix": cm.tolist(), "total_params": int(num_params),
        "avg_latency_ms": float(inf_time * 1000),
    }
    print(f"{name}: acc={acc:.4f} macroF1={macro_f1:.4f} AUC={auc_val:.4f}")

with open(f"{DRIVE_ROOT}/results/metrics_summary_v2.json", "w") as f:
    json.dump({"models": all_results}, f, indent=2)
print("Saved metrics_summary_v2.json to Drive.")

## 8. Pairwise McNemar tests

In [ ]:
mcnemar_results = {}
names = MODELS_TO_TRAIN
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        n1, n2 = names[i], names[j]
        c1, c2 = (pred_labels[n1] == y_true), (pred_labels[n2] == y_true)
        both_correct = int(np.sum(c1 & c2))
        a_right_b_wrong = int(np.sum(c1 & ~c2))
        a_wrong_b_right = int(np.sum(~c1 & c2))
        both_wrong = int(np.sum(~c1 & ~c2))
        disc = a_right_b_wrong + a_wrong_b_right
        stat = ((abs(a_right_b_wrong - a_wrong_b_right) - 1.0) ** 2) / disc if disc > 0 else 0.0
        pval = float(scistats.chi2.sf(stat, df=1))
        mcnemar_results[f"{n1}_vs_{n2}"] = {
            "contingency_table": [[both_correct, a_right_b_wrong], [a_wrong_b_right, both_wrong]],
            "statistic": float(stat), "p_value": pval, "significant_at_05": pval < 0.05
        }
        print(f"{n1} vs {n2}: chi2={stat:.2f}, p={pval:.2e}", "(sig)" if pval < 0.05 else "(not sig)")

with open(f"{DRIVE_ROOT}/results/mcnemar_v2.json", "w") as f:
    json.dump(mcnemar_results, f, indent=2)

## 9. Quick figures + zip everything for download

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, names):
    cm = np.array(all_results[name]["confusion_matrix"])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0, 1]); ax.set_yticklabels(CLASS_NAMES)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center", color="white" if cm[r, c] > cm.max()/2 else "black")
    ax.set_title(f"{name}\nAcc: {all_results[name]['accuracy']*100:.1f}%")
fig.tight_layout()
fig.savefig(f"{DRIVE_ROOT}/results/confusion_matrices_v2.png", dpi=300)
plt.show()

import shutil as sh
sh.make_archive("/content/leukemia_rerun_results", "zip", f"{DRIVE_ROOT}/results")
from google.colab import files as colab_files
colab_files.download("/content/leukemia_rerun_results.zip")
print("Send this back and I'll update the paper with the fully-converged numbers.")